<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/LLM/%E1%84%85%E1%85%A2%E1%86%BC%E1%84%8E%E1%85%A6%E1%84%8B%E1%85%B5%E1%86%AB%E1%84%8B%E1%85%B3%E1%84%85%E1%85%A9_RAG_%E1%84%89%E1%85%B5%E1%84%8C%E1%85%A1%E1%86%A8%E1%84%92%E1%85%A1%E1%84%80%E1%85%B5_%5B%E1%84%91%E1%85%B3%E1%84%85%E1%85%A9%E1%84%8C%E1%85%A6%E1%86%A8%E1%84%90%E1%85%B3%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 랭체인으로 RAG 시작하기

해당 레슨은 Langchain으로 RAG를 구현하기 위해 필요한 각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다

## Step 0 : 설치와 준비

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters

In [3]:
import os

In [4]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [5]:
#! curl ipinfo.io

In [6]:
from langchain_openai import ChatOpenAI
import os
from google.colab import userdata

# OpenAI API를 사용하는 설정으로 변경
llm = ChatOpenAI(
    model="gpt-6-sol",
)

In [7]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

## Step 1 : Document Loaders 사용해보기

In [8]:
# 실습용 PDF(헤르만 헤세 《데미안》 영문판, 퍼블릭 도메인, 약 5.7MB) 를 내려받습니다. 파일이 이미 있고 비어 있지 않으면 건너뜁니다.
import os
if not os.path.exists("Demian.pdf") or os.path.getsize("Demian.pdf") == 0:
    !gdown 1oYqBihNheRgOH2JQ0zaXXLHuSuY27IPq -O Demian.pdf
!ls -lh Demian.pdf

-rw-r--r-- 1 root root 5.5M Sep 22 18:11 Demian.pdf


In [9]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

/tmp/ipykernel_2198/128205925.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [11]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

In [12]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

```두 개의 세상
결국, 지나친 긴장감에 휩싸여 나는 말을 꺼내기 시작했다. 나는
내가 주인공으로 등장하는 긴 강도 이야기를 지어냈다. 어느 날 밤,
제분소 옆 구석에서 친구와
나는 자루 한 가득 사과를 훔쳤는데, 그건
평범한 사과가 아니라 피핀 사과, 그것도 최고급
황금빛 피핀 사과였다. 나는 그 순간의 위험에서 벗어나기 위해
그 이야기에 의지하고 있었고, 이야기를 지어내고
전하는 데 아무런 어려움도 느끼지 못했다. 말이 너무 빨리 끊겨서
어쩌면 더 나쁜 상황에 휘말리지 않기 위해, 나는
이야기하는 능력을 마음껏 발휘했다. 나는 우리 중 한 명이
항상 경계를 서고, 다른 한 명은 나무 위에 앉아
사과를 아래로 던졌다고 말했고, 자루가 너무 무거워져서
결국 자루를 열어 절반은 남겨두고 와야 했지만,
30분 후에 다시 돌아와 그 사과들도
다 가져왔다고 했다.
이야기가 끝날 때쯤 박수라도 받을 수 있기를 바랐는데; 나는
결국 내 이야기의 열기에 휩쓸려
말솜씨에 취해 열을 올리고 있었다. 두 어린 소년은 침묵하며
기다리고 있었지만, 프란츠 크로머는 좁은 눈으로
나를 뚫어지게 쳐다보았다. “그 이야기가 사실이야?” 그가
위협적인 어조로 물었다.
“네,” 내가 대답했다.
“정말, 진심으로?”
“네, 정말로 사실이에요,” 나는
속으로는 두려움에 숨이 막힐 지경이었지만, 도전적으로 단언했다.
“맹세할 수 있겠어?”
나는 무척 두려웠지만 망설임 없이 ‘네’라고 대답했다.
“가슴에 손을 얹고?”
“가슴에 손을 얹고,”
“좋아,” 그가 말하고는 등을 돌렸다.
나는 이 모든 상황이 아주 만족스러웠고 기뻤다.
15

In [13]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [14]:
!pip install tiktoken

## Step3 : TextEmbedding 사용해보기

In [15]:
import openai

In [16]:
client = openai.OpenAI()

In [17]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-3-small
text-embedding-3-large
text-embedding-ada-002


In [18]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

## Step4 : VectorStore 사용해보기

In [19]:
!pip install chromadb

In [20]:
!pip install langchain-chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)


In [21]:
from langchain_chroma import Chroma

In [22]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!

In [24]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    return len(tokenizer.encode(text))

In [25]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

Chroma에 임베딩 시킵니다

In [26]:
db = Chroma.from_documents(docs, embedding_model)

이제 쿼리를 날려보겠습니다

In [27]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [28]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

```메스꺼움을 느끼며 나는 데미안의 표정을 눈치챘다.
그는 앞으로 나서지 않고 맨
뒤에 서서, 여느 때처럼 우아하고 여유로운 모습이었다. 그의
시선은 말의 머리를 향하고 있는 듯했고, 거기에는 다시금
깊고 고요하며, 거의 광신적이면서도 열정적인
몰입이 드러나 있었다. 나는 잠시 동안 그를 빤히 쳐다보지 않을 수 없었고,
바로 그때 내 깊은 의식 속에서 매우
기이한 감각을 느꼈다. 나는
데미안의 얼굴을 보았고, 그것이 소년의 얼굴이 아니라
남자의 얼굴이라는 것을 깨달았다. 그리고 나서 나는,
아니 오히려 깨닫게 되었는데, 그것이 사실 남자의 얼굴도 아니라는 것을; 그 얼굴에는 뭔가
무언가 다른 점이 있었고, 거의 여성적인 요소가
담겨 있었다. 그리고
그 순간 그의 얼굴은 남성적이지도
아이 같지도 않았고, 늙지도 젊지도 않은, 마치 백 년
이나 된 듯한, 거의 시간을 초월한 듯하며 우리 시대와는
다른 역사적 시대의 흔적을 띠고 있는 것처럼 보였다. 동물이나
나무, 혹은 별들이 그런 모습일지도 모른다. 물론 그 당시 나는
지금, 성인이 되어 이 글을 쓰면서 느끼는 것과
정확히 같은 감정을 느끼지는 못했지만, 대략 그런 느낌이었다. 어쩌면 그는
잘생겼을지도 모르고, 어쩌면 내가 그를 매력적으로 느꼈을지도 모르고, 어쩌면
그가 나를 밀어내기도 했을지 모른다. 그조차도 확신할 수 없었다. 내가
본 것은 그가 우리 나머지 사람들과는 달랐다는 것,
그가 마치 동물이나 영혼, 혹은 환영 같았다는 것뿐이었다. 그를
설명할 수 있는 말은 오직 그가 달랐다는 것,
우리 나머지 사람들과는 상상조차 할 수 없을 만큼 달랐다는 것뿐이다. 그 외의 모든 것에 대해서는
기억이 전혀 나지 않으며, 어쩌면 내가
설명한 것조차 어느 정도는 나중에
받은 인상에서 추론한 것일지도 모른다.
내가 몇 살 더 먹은 후에야 비로소
그와 가까워질 수 있었다. 일반적인 관습과는 달리,
데미안은 다른 소년들과 함께 견진성사를 받지 않았는데,

## Step5 : Retriever 사용해보기

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.

In [29]:
!pip install -U langchain langchain-classic

In [30]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [31]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-6-sol",
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [32]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [33]:
query = "how demian looks like? "
result = qa(query)

/tmp/ipykernel_2198/670885640.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian looks elegant and at ease, with brown hair and a face the narrator finds strikingly hard to place. It seems handsome but also almost timeless—neither quite boyish nor adult, masculine nor feminine. At times his pale, still face looks like a stone mask, though his gaze is intensely focused.

마크다운 형식으로 출력해봅니다

In [34]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian looks elegant and at ease, with brown hair and a face the narrator finds strikingly hard to place. It seems handsome but also almost timeless—neither quite boyish nor adult, masculine nor feminine. At times his pale, still face looks like a stone mask, though his gaze is intensely focused.

In [35]:
translation_request = llm.invoke(f"Translate the following English text to Korean: {result['result']}")
display(Markdown(translation_request.content))

데미안은 갈색 머리에 우아하고 여유로워 보인다. 화자는 그의 얼굴을 어떻게 묘사해야 할지 좀처럼 알 수 없다. 잘생긴 듯하면서도 시대를 초월한 듯한 인상으로, 소년 같지도 완전히 어른 같지도 않고, 남성적이지도 여성적이지도 않다. 때로 그의 창백하고 미동 없는 얼굴은 돌로 만든 가면처럼 보이지만, 시선만큼은 강렬하게 한곳에 집중되어 있다.

데미안은 갈색 머리에 우아하고 여유로워 보인다. 화자는 그의 얼굴을 어떻게 묘사해야 할지 좀처럼 알 수 없다. 잘생긴 듯하면서도 시대를 초월한 듯한 인상으로, 소년 같지도 완전히 어른 같지도 않고, 남성적이지도 여성적이지도 않다. 때로 그의 창백하고 미동 없는 얼굴은 돌로 만든 가면처럼 보이지만, 시선만큼은 강렬하게 한곳에 집중되어 있다.

1. 페이지 수 출력

In [36]:
pdf_pages = loader.load()
print("PDF 페이지 수:", len(pdf_pages))

PDF 페이지 수: 182


2. Demian chunk 수 출력

In [38]:
demian_chunks = text_splitter.split_documents(pages)
print("Demian chunk 수:", len(demian_chunks))

Demian chunk 수: 182


3. Demian chunk의 임베딩 차원 출력

In [39]:
sample_vector = embedding_model.embed_documents(
    [demian_chunks[0].page_content]
)[0]

print("Demian chunk 임베딩 차원:", len(sample_vector))

Demian chunk 임베딩 차원: 1536


4. 기존 retriever의 검색 결과 출력

In [40]:
retrieval_question = "Why did Damian tell the story about stealing the apple?"
retrieved_docs = qa.retriever.invoke(retrieval_question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n[검색 결과 {i} | PDF {doc.metadata['page'] + 1}쪽]")
    print(doc.page_content)


[검색 결과 1 | PDF 13쪽]
TWO WORLDS 
to him so that I couldn't avoid l~king into h~ face at 
dose range. He had an evil gleam in his eyes and he 
gave an ugly laugh. His face was full of cruelty and 
sense of power. 
"Yes, kid, I can tell you who owns the orchard. I've 
known for a long time that people have been stealing 
apples and I also know that the man in question said he 
would give two marks reward to anyone who could tell 
him who stole the fruit." 
"Heavens I " I exclaimed. "But you wouldn't let on to 
him?" 
I felt that it would be futile to appeal to his sense of 
honour. He belonged to the 'other' world; betrayal was 
no aime as far as he was concerned. That much was 
dear to me. The people of the •other' world were not 
like us in these matters. 
"Not let on! My dear fellow, do you think I can mint 
my own money and produce a couple of marks out of a 
hat? I'm poor. I haven't a rich father like you and if I 
can earn two marks I've got to earn them. He might even 
give me mor

**[검색 결과 1 | PDF 13쪽 — 「두 세계」]**

…그와 너무 가까이 있어서 그의 얼굴을 바로 앞에서 바라볼 수밖에 없었다. 그의 눈에는 사악한 빛이 번뜩였고, 그는 기분 나쁘게 웃었다. 얼굴에는 잔인함과 힘을 쥐었다는 만족감이 가득했다.

“그래, 꼬마야. 그 과수원이 누구 것인지 내가 알려 줄 수 있어. 사람들이 사과를 훔쳐 간다는 건 오래전부터 알고 있었지. 그 주인이 과일을 훔친 사람이 누군지 알려 주면 사례금으로 2마르크를 주겠다고 했다는 것도 알아.”

“맙소사!” 내가 외쳤다. “하지만 그 사람한테 말하지는 않을 거지?”

그의 의리에 호소해 봐야 소용없으리라는 생각이 들었다. 그는 ‘다른’ 세계에 속한 사람이었다. 그에게 배신은 죄가 아니었다. 적어도 그것만은 분명했다. ‘다른’ 세계의 사람들은 이런 일에서 우리와 달랐다.

“말하지 말라고? 이봐, 내가 돈을 찍어 내거나 모자 속에서 2마르크를 꺼낼 수 있을 것 같아? 나는 가난해. 너처럼 부자 아버지가 있는 것도 아니고. 2마르크를 벌 수 있다면 벌어야지. 어쩌면 그 사람이 더 줄 수도 있고.”

그는 갑자기 나를 떠났다. 우리 집 복도에서는 더 이상 평온함과 안전함의 냄새가 나지 않았다. 세상이 내 주위에서 무너지고 있었다. 그는 나를 범죄자로 고발할 것이다. 사람들이 아버지에게 알릴 것이고, 어쩌면 경찰이 찾아올지도 모른다.

온갖 혼란의 공포가 나를 위협했다. 내 앞에 놓인 상황은 끔찍하고 위험했다. 실제로 도둑질하지 않았다는 사실은 사소한 문제에 불과했다. 나는 훔쳤다고 맹세했으니까.

하늘에 계신 하느님! 눈에 눈물이 고였다. 돈을 주고 이 상황에서 벗어나야 한다는 생각에 필사적으로 주머니를 더듬었다. 사과도, 주머니칼도 없었다. 아무것도 없었다.

---

**[검색 결과 2 | PDF 30쪽 — 「Demian」]**

“그래. 그러니까 나는 Cain의 이야기를 다르게 해석할 수 있다고 생각해.” 그가 말을 이었다. “우리가 배우는 이야기들은 대부분 근거가 있고 진실을 담고 있어. 하지만 선생님들이 보는 것과 다른 관점에서 바라볼 수도 있지. 그러면 훨씬 더 말이 돼.

예를 들어, 사람들이 설명하는 Cain과 그의 이마에 찍힌 표식은 아무래도 납득하기 어렵잖아. 너도 그렇지 않아? 누군가 다투다가 자기 형제를 죽일 수는 있겠지. 그 뒤 겁을 먹고 비굴하게 굴 수도 있고. 하지만 그 비겁함 때문에 특별한 ‘표식’을 받았는데, 그것이 일종의 보호 장치가 되어 다른 사람들에게 두려움을 불러일으킨다는 건 참 이상하지.”

“그렇네.” 나는 그 이야기에 흥미가 생겨 말했다. “그런데 어떻게 다르게 해석할 수 있다는 거야?”

그가 다시 내 어깨를 툭 쳤다.

“아주 간단해. 실제로 일어났던 일, 그리고 이 이야기 전체가 생겨난 배경에는 그 ‘표식’이 있었던 거야. 얼굴에 다른 사람들을 두렵게 하는 무언가를 지닌 사람이 있었던 거지. 사람들은 감히 그에게 손을 대지 못했어. 그와 그의 자식들은 사람들에게 강한 인상을 주었지.

그의 이마에 우편 소인처럼 실제 표식이 찍혀 있지 않았다는 건 거의 확실해! 현실은 그렇게 단순하지 않거든. 그보다는 거의 알아보기 힘든 어떤 특징이 있었겠지. 사람들이 익숙하게 보던 것보다 조금 더 지적이고 침착한 눈빛 같은 것 말이야.

그 사람에게는 힘이 있었고, 모두가 그를 두려워하며 우러러보았어. 그에게는 ‘표식’이 있었던 거야. 그걸 어떻게 설명하든 상관없어.

사람들은 언제나 자신을 편안하게 하고, 자신이 옳다고 느끼게 해 주는 것을 원해. 그들은 Cain의 자식들을 두려워했어. 그 자식들도 표식을 지니고 있었거든. 그래서 사람들은 그 표식의 실제 의미를 정반대로 해석했지. 그 표식을 가진 사람들은 이상한 사람들이라고 말했어. 실제로 그들은…”

---

**[검색 결과 3 | PDF 37쪽 — 「Cain」]**

…이 이야기는 그런 사람들을 위한 것이 아니다. 나는 인간의 본성을 더 잘 이해하는 사람들에게 이 이야기를 들려주고 있다.

자기 감정의 일부를 생각으로 옮기는 법을 배운 어른은 아이에게서는 그러한 생각을 발견하지 못하고, 결국 아이가 겪은 경험 자체까지 부정한다. 하지만 나는 그때만큼 깊이 느끼고 고통받은 적이 거의 없었다.

비가 내리던 어느 날, 나를 괴롭히던 그가 시내 광장으로 나오라고 명령했다. 나는 젖은 나무에서 여전히 떨어지고 있던 축축한 밤나무 잎들 사이에서 발을 끌며 서서 기다렸다.

돈은 없었지만, Kromer에게 무엇이라도 줄 수 있도록 케이크 두 조각을 따로 챙겨 두었다. 나는 구석에 서서 그를 기다리는 일에 익숙해져 있었다. 아주 오랫동안 기다리는 경우도 많았다. 피할 수 없는 일을 견디는 법을 배우듯, 나는 그것을 참아 냈다.

마침내 Kromer가 다가왔다. 그는 오래 머물지 않았다. 내 옆구리를 몇 번 쿡쿡 찌르고 웃더니 케이크를 가져갔다. 축축한 담배 한 개비를 건네기까지 했다. 평소보다 친근하게 굴었다.

“아, 맞다.” 그가 떠나면서 말했다. “잊기 전에 말해 두는데, 다음에는 네 누나도 데려와. 큰누나 말이야. 그런데 이름이 뭐지?”

나는 그 말뜻을 이해하지 못했고, 아무 대답도 하지 않았다. 놀란 눈으로 그를 바라보았다.

“못 알아들었어? 네 누나를 데려오라고.”

“응, Kromer. 하지만 그건 안 돼. 허락도 못 받을 거고, 어차피 누나도 오지 않을 거야.”

그의 제안은 놀랍지 않았다. 나는 그것을 있는 그대로…

5. 기존 답변 1개와 새 질문 2개의 답변·출처 출력

In [41]:
qa_results = [("how demian looks like?", result)]

for question in [
    "Why did Damian tell the story about stealing the apple?",
    "How does Sinclair first meet Demian?",
]:
    qa_results.append((question, qa.invoke({"query": question})))

for n, (question, answer) in enumerate(qa_results, 1):
    print(f"\n=== 질문 {n}: {question} ===")
    print("답변:", answer["result"])

    for i, source in enumerate(answer["source_documents"], 1):
        print(f"\n[출처 {i} | PDF {source.metadata['page'] + 1}쪽]")
        print(source.page_content)

Demian didn’t tell the apple-stealing story. Sinclair claimed he had stolen apples, though he hadn’t, and Kromer used that claim to threaten him.I don’t know from the provided excerpts. They describe Sinclair reuniting with Demian, not how they first met.
=== 질문 1: how demian looks like? ===
답변: Demian looks elegant and at ease, with brown hair and a face the narrator finds strikingly hard to place. It seems handsome but also almost timeless—neither quite boyish nor adult, masculine nor feminine. At times his pale, still face looks like a stone mask, though his gaze is intensely focused.

[출처 1 | PDF 54쪽]
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt 


---

**질문 1: Demian은 어떻게 생겼나요?**

**답변:** Demian은 품위 있고 여유로워 보이며, 갈색 머리카락을 지녔습니다. 화자는 그의 얼굴을 어떤 유형으로도 쉽게 규정하기 어렵다고 느낍니다. 잘생긴 듯하면서도 시간을 초월한 모습으로, 소년과 어른, 남성과 여성의 특징이 함께 드러납니다. 때로는 창백하고 움직임 없는 얼굴이 돌로 만든 가면처럼 보이지만, 시선에는 강렬한 집중력이 담겨 있습니다.

**[출처 1 | PDF 54쪽]**

구역질이 나는 가운데, 나는 Demian의 표정을 살펴보았다. 그는 앞쪽으로 나서지 않고 맨 뒤에 서 있었으며, 평소처럼 품위 있고 느긋해 보였다. 그의 시선은 말의 머리를 향하는 듯했다. 그 눈빛에는 다시금 깊고 고요한 몰입이 드러나 있었다. 거의 광적이면서도 열정적인 몰입이었다.

나는 한동안 그를 바라보지 않을 수 없었다. 바로 그때, 의식의 아득한 곳에서 무척 기이한 느낌이 일어나는 것을 느꼈다. Demian의 얼굴을 보며, 그것이 소년의 얼굴이 아니라 어른 남자의 얼굴이라는 생각이 들었다. 그러다가 그것이 사실은 어른 남자의 얼굴도 아니라는 것을 보았다. 아니, 깨달았다고 해야 할 것이다. 그 얼굴에는 무언가 다른 것이, 거의 여성적인 요소가 깃들어 있었다.

그 순간 그의 얼굴은 남성적이지도, 어린아이 같지도 않았다. 늙지도 젊지도 않았지만 백 년쯤 된 듯했고, 거의 시간을 초월한 채 우리가 사는 시대와는 다른 역사적 시대들의 흔적을 지니고 있었다. 동물이나 나무, 별들이 그런 모습을 하고 있을지도 몰랐다.

물론 당시의 나는 그런 것을 알지 못했다. 어른이 된 지금 이렇게 쓰고 있는 감정을 그때 정확히 느낀 것은 아니었지만, 그와 비슷한 무엇인가를 느꼈다. 어쩌면 그는 잘생겼고, 나는 그에게 매력을 느꼈을지도 모른다. 동시에 그에게 거부감을 느꼈을 수도 있다. 그조차 확신할 수 없었다.

내가 알 수 있었던 것은 그가 우리와 다르다는 사실뿐이었다. 그는 동물이나 영혼, 혹은 어떤 형상과도 같았다. 그가 우리와 달랐다는 것, 상상할 수 없을 만큼 달랐다는 것 외에는 그를 설명할 수 없다. 그 밖의 모든 일에 대한 기억은 텅 비어 있다. 어쩌면 지금 묘사한 것조차 어느 정도는 나중에 받은 인상에서 끌어온 것인지도 모른다.

내가 그와 가까이 지내게 된 것은 몇 년 더 나이가 든 뒤였다. 일반적인 관례와 달리, Demian은 다른 소년들과 함께 견진성사를 받지 않았는데…

**[출처 2 | PDF 71쪽 — 「도둑」]**

…나는 넋을 잃은 채 그 창백한 돌가면에 시선을 고정했다. 그리고 이것이야말로 진짜 Demian이라고 느꼈다! 이전에 나와 어울려 다니며 이야기를 나누던 모습은 그의 절반에 불과했다. 그는 잠시 어떤 역할을 맡아 그 역할에 자신을 맞추고, 나를 기쁘게 해 주려고 놀이에 함께해 주고 있었던 것이다.

진짜 Demian은 이런 모습이었다. 홀로 떨어져 있고, 아득히 오래되었으며, 동물 같고 돌 같았다. 아름답고 차갑고 죽어 있는 듯했지만, 그 안에는 섬뜩할 만큼 은밀한 생명이 깃들어 있었다. 그의 주위에는 고요한 공허의 기운, 천상의 공간과 에테르, 고독한 죽음이 감돌고 있었다!

‘이제 그는 완전히 자기 안으로 물러났구나.’ 나는 몸서리치며 생각했다. 그렇게 외롭게 느껴진 적은 없었다. 나는 그의 내면에 조금도 참여할 수 없었다. 그는 내 손이 닿지 않는 곳에 있었다. 세상에서 가장 먼 외딴섬에 있다 해도 지금보다는 나와 가까웠을 것이다.

이것을 알아차린 사람이 나뿐이라는 사실을 좀처럼 이해할 수 없었다. 모두가 그를 바라보고, 모두가 몸서리치고 있으리라고 느꼈다! 하지만 아무도 그에게 관심을 기울이지 않았다. 그는 조각된 형상처럼, 우상처럼 뻣뻣하게 앉아 있었다. 파리 한 마리가 그의 이마에 내려앉아 코와 입술 위를 기어 다녔는데도 얼굴 근육 하나 움직이지 않던 모습을 나는 잊을 수 없다.

그는 어디로 물러난 것일까? 무엇을 생각하고, 무엇을 느끼고 있었을까? 천국에 있었을까, 지옥에 있었을까? 답할 수 없는 질문이었다.

수업이 끝날 무렵, 그가 다시 숨을 쉬며 생기를 되찾고 내 눈과 시선을 마주쳤을 때, 그의 모습은 이전과 다르지 않았다. 그는 어디에서 돌아온 것일까? 어디에 있었던 것일까?

그는 피곤해 보였다. 얼굴에는 혈색이 돌아왔고 손도 다시 움직였다. 하지만 갈색 머리카락은 어쩐지 윤기를 잃고 생기 없어 보였다…

**[출처 3 | PDF 128쪽]**

…강렬한 열정으로. 그러자 곧 내 꿈이 다시 떠올랐다. 우리 집의 대문과 문장, 어머니와 낯선 여인이었다. 그 여인의 얼굴은 초자연적일 만큼 또렷하게 보였기에, 나는 바로 그날 저녁 기억만으로 그녀의 초상화를 그릴 수 있었다.

며칠 뒤 그림이 완성되었다. 틈틈이 십 분 정도 시간이 날 때마다 거의 무의식적으로 색을 칠한 그림이었다. 나는 그날 저녁 그림을 벽에 걸고, 책상 램프를 그 앞에 옮겨 놓았다. 그리고 결판을 내야 할 영혼을 마주하듯 그림 앞에 섰다.

그 얼굴은 이전 그림과 닮아 있었다. 내 친구 Demian과도 비슷했고, 어떤 점에서는 나 자신을 닮기도 했다. 한쪽 눈이 다른 쪽보다 눈에 띄게 높았고, 그녀의 시선은 운명적인 눈빛으로 나를 넘어 저 너머를 바라보고 있었다.

그녀 앞에 서자 긴장으로 피가 차갑게 식는 듯했다. 나는 그림에 질문하고, 비난하고, 어루만지고, 기도했다. 그림을 어머니, 연인, 창녀, 매춘부라고 불렀고, Abraxas라고 불렀다.

그사이 Pistorius의 말들이 머릿속에 떠올랐다. 아니, Demian의 말이었을까? 언제 들었던 말인지는 기억나지 않았지만, 다시 그 목소리가 들리는 듯했다. 그것은 하느님의 천사와 씨름한 Jacob, 그리고 그가 했던 “저를 축복해 주시기 전에는 놓아드리지 않겠습니다”라는 말에 관한 이야기였다.

램프 불빛 아래의 얼굴은 내가 말을 할 때마다 변했다. 밝게 빛났다가 어둡고 불길해졌다. 창백한 눈꺼풀이 흐릿한 눈을 덮었다가 다시 열렸고, 눈은 사랑스러운 시선을 보냈다. 그것은 여인이었다가 남자가 되고, 소녀였다가 어린아이가 되고, 동물이 되었다. 그러다가 작은 점으로 녹아 사라졌다가 다시 크고 또렷해졌다.

마침내 나는 눈을 감고 싶은 강한 충동에 몸을 맡겼다. 그러자 이제…

---

**질문 2: Damian은 왜 사과를 훔쳤다는 이야기를 했나요?**

**답변:** 사과를 훔쳤다는 이야기를 한 사람은 Demian이 아니었습니다. Sinclair는 실제로 사과를 훔치지 않았는데도 훔쳤다고 말했고, Kromer는 그 말을 이용해 그를 협박했습니다.

**[출처 1 | PDF 13쪽 — 「두 세계」]**

…그와 너무 가까워서 그의 얼굴을 바로 앞에서 바라볼 수밖에 없었다. 그의 눈에는 사악한 빛이 번뜩였고, 그는 기분 나쁘게 웃었다. 얼굴에는 잔인함과 힘을 쥐었다는 만족감이 가득했다.

“그래, 꼬마야. 그 과수원이 누구 것인지 내가 알려 줄 수 있어. 사람들이 사과를 훔쳐 간다는 건 오래전부터 알고 있었지. 그리고 그 주인이 과일을 훔친 사람이 누군지 알려 주면 사례금으로 2마르크를 주겠다고 했다는 것도 알아.”

“맙소사!” 내가 외쳤다. “하지만 그 사람한테 말하지는 않을 거지?”

그의 의리에 호소해 봐야 소용없으리라는 생각이 들었다. 그는 ‘다른’ 세계에 속한 사람이었다. 그에게 배신은 죄가 아니었다. 적어도 그것만은 분명했다. ‘다른’ 세계의 사람들은 이런 일에서 우리와 달랐다.

“말하지 말라고? 이봐, 내가 돈을 찍어 내거나 모자 속에서 2마르크를 꺼낼 수 있을 것 같아? 나는 가난해. 너처럼 부자 아버지가 있는 것도 아니고. 2마르크를 벌 수 있다면 벌어야지. 어쩌면 그 사람이 더 줄 수도 있고.”

그는 갑자기 나를 떠났다. 우리 집 복도에서는 더 이상 평온함과 안전함의 냄새가 나지 않았다. 세상이 내 주위에서 무너지고 있었다. 그는 나를 범죄자로 고발할 것이다. 사람들이 아버지에게 알릴 것이고, 어쩌면 경찰이 찾아올지도 모른다.

온갖 혼란의 공포가 나를 위협했다. 내 앞에 놓인 상황은 끔찍하고 위험했다. 실제로 도둑질하지 않았다는 사실은 사소한 문제에 불과했다. 나는 훔쳤다고 맹세했으니까.

하늘에 계신 하느님! 눈에 눈물이 고였다. 돈을 주고 이 상황에서 벗어나야 한다는 생각에 필사적으로 주머니를 더듬었다. 사과도, 주머니칼도 없었다. 아무것도 없었다.

**[출처 2 | PDF 30쪽]**

“그래. 그러니까 나는 Cain의 이야기를 다르게 해석할 수 있다고 생각해.” 그가 말을 이었다. “우리가 배우는 이야기들은 대부분 근거가 있고 진실을 담고 있어. 하지만 선생님들이 보는 것과 다른 관점에서 바라볼 수도 있지. 그러면 훨씬 더 말이 돼.

예를 들어, 사람들이 설명하는 Cain과 그의 이마에 찍힌 표식은 아무래도 납득하기 어렵잖아. 너도 그렇지 않아? 누군가 다투다가 자기 형제를 죽일 수는 있겠지. 그 뒤 겁을 먹고 비굴하게 굴 수도 있고. 하지만 그 비겁함 때문에 특별한 ‘표식’을 받았는데, 그것이 일종의 보호 장치가 되어 다른 사람들에게 두려움을 불러일으킨다는 건 참 이상하지.”

“그렇네.” 나는 그 이야기에 흥미가 생겨 말했다. “그런데 어떻게 다르게 해석할 수 있다는 거야?”

그가 다시 내 어깨를 툭 쳤다.

“아주 간단해. 실제로 일어났던 일, 그리고 이 이야기 전체가 생겨난 배경에는 그 ‘표식’이 있었던 거야. 얼굴에 다른 사람들을 두렵게 하는 무언가를 지닌 사람이 있었던 거지. 사람들은 감히 그에게 손을 대지 못했어. 그와 그의 자식들은 사람들에게 강한 인상을 주었지.

그의 이마에 우편 소인처럼 실제 표식이 찍혀 있지 않았다는 건 거의 확실해! 현실은 그렇게 단순하지 않거든. 그보다는 거의 알아보기 힘든 어떤 특징이 있었겠지. 사람들이 익숙하게 보던 것보다 조금 더 지적이고 침착한 눈빛 같은 것 말이야.

그 사람에게는 힘이 있었고, 모두가 그를 두려워하며 우러러보았어. 그에게는 ‘표식’이 있었던 거야. 그걸 어떻게 설명하든 상관없어.

사람들은 언제나 자신을 편안하게 하고, 자신이 옳다고 느끼게 해 주는 것을 원해. 그들은 Cain의 자식들을 두려워했어. 그 자식들도 표식을 지니고 있었거든. 그래서 사람들은 그 표식의 실제 의미를 정반대로 해석했지. 그 표식을 가진 사람들은 이상한 사람들이라고 말했어. 실제로 그들은…”

**[출처 3 | PDF 37쪽 — 「Cain」]**

…이 이야기는 그런 사람들을 위한 것이 아니다. 나는 인간의 본성을 더 잘 이해하는 사람들에게 이 이야기를 들려주고 있다.

자기 감정의 일부를 생각으로 옮기는 법을 배운 어른은 아이에게 그런 생각이 없다고 여기고, 결국 아이가 겪은 경험 자체까지 부정한다. 하지만 나는 그때만큼 깊이 느끼고 고통받은 적이 거의 없었다.

비가 내리던 어느 날, 나를 괴롭히던 그가 시내 광장으로 나오라고 명령했다. 나는 젖은 나무에서 여전히 떨어지고 있던 축축한 밤나무 잎들 사이에서 발을 끌며 서서 기다렸다.

돈은 없었지만, Kromer에게 무엇이라도 줄 수 있도록 케이크 두 조각을 따로 챙겨 두었다. 나는 구석에 서서 그를 기다리는 일에 익숙해져 있었다. 아주 오랫동안 기다리는 경우도 많았다. 피할 수 없는 일을 견디는 법을 배우듯, 나는 그것을 참아 냈다.

마침내 Kromer가 다가왔다. 그는 오래 머물지 않았다. 내 옆구리를 몇 번 쿡쿡 찌르고 웃더니 케이크를 가져갔다. 축축한 담배 한 개비를 건네기까지 했다. 평소보다 친근하게 굴었다.

“아, 맞다.” 그가 떠나면서 말했다. “잊기 전에 말해 두는데, 다음에는 네 누나도 데려와. 큰누나 말이야. 그런데 이름이 뭐지?”

나는 그 말뜻을 이해하지 못했고, 아무 대답도 하지 않았다. 놀란 눈으로 그를 바라보았다.

“못 알아들었어? 네 누나를 데려오라고.”

“응, Kromer. 하지만 그건 안 돼. 허락도 못 받을 거고, 어차피 누나도 오지 않을 거야.”

그의 요구가 뜻밖이라고 느껴지지는 않았다. 나는 그것이 어떤 의미인지 알았다…

---

**질문 3: Sinclair는 Demian을 처음 어떻게 만나나요?**

**답변:** 제공된 발췌문만으로는 알 수 없습니다. 이 발췌문들은 Sinclair가 Demian과 다시 만나는 장면을 묘사하며, 두 사람이 처음 어떻게 만났는지는 설명하지 않습니다.

**[출처 1 | PDF 146쪽]**

“일본에서도 사정이 더 낫지는 않을 것 같군요. 안락한 난롯가로 곧장 달려가지 않는 사람은 세계 어디에서나 드물지요. 여기에도 몇 사람은 있습니다.”

그가 하는 말 한마디 한마디가 내 안을 꿰뚫을 때마다 기쁨과 놀라움이 뒤섞였다. 나는 그 목소리의 주인을 알았다. Demian이었다.

나는 돌풍이 부는 밤, 어두운 골목을 따라 그와 일본인의 뒤를 쫓으며 두 사람의 대화를 들었다. Demian의 목소리는 내 귀에 음악처럼 들렸다. 익숙한 울림, 익숙한 확신과 평온함, 그리고 예전에 나를 사로잡았던 힘이 그대로 담겨 있었다. 모든 것이 다시 제자리를 찾았다. 나는 그를 찾은 것이다.

그들이 교외의 한 거리 끝에 이르자 일본인은 작별 인사를 하고 어느 집의 문을 열었다. Demian은 왔던 길을 되돌아왔다. 나는 거리 한가운데 멈춰 서서 그를 기다리고 있었다.

그가 탄력 있는 걸음과 곧은 자세로 다가오는 모습을 보자 심장이 빠르게 뛰었다. 그는 엷은 황갈색 비옷을 입고 있었고, 가느다란 지팡이가 팔에 매달려 흔들리고 있었다.

그는 거의 내 앞에 이를 때까지 일정한 걸음으로 다가오다가 모자를 벗었다. 그러자 익숙한 그 지적인 얼굴이 더 또렷하게 보였다. 굳은 의지를 드러내는 입매와 넓은 이마의 독특한 광채도 보였다.

“Demian!” 내가 외쳤다.

그가 내게 손을 내밀었다.

“너였구나, Sinclair! 너를 기다리고 있었어.”

“내가 여기 있는 줄 알고 있었어?”

“정확히 알았던 건 아니지만, 네가 여기 있기를 바랐지. 너를 본 건 오늘 밤이 처음이야. 계속 우리 뒤를 따라왔더구나.”

“그럼 나를 바로 알아봤어?”

**[출처 2 | PDF 182쪽]**

“…또 하나 있어. Eva 부인이 네가 몹시 힘들어질 때면, 나에게 주신 입맞춤을 네게 전해 달라고 하셨어… 눈을 감아, Sinclair.”

나는 그 말에 따라 눈을 감았다. 줄어들 줄 모르는 핏방울이 맺힌 내 입술 위로 입맞춤이 가볍게 스치는 것을 느꼈다. 그러고는 잠이 들었다.

다음 날 아침 사람들이 나를 깨웠다. 상처를 치료받아야 했다. 마침내 정신이 완전히 들었을 때, 나는 재빨리 옆 매트리스 쪽으로 고개를 돌렸다. 그 위에는 한 번도 본 적 없는 낯선 사람이 누워 있었다.

상처를 치료하는 일은 고통스러웠다. 그 뒤 내게 일어난 다른 모든 일도 마찬가지였다.

하지만 그런 수많은 순간에, 내가 열쇠를 찾아 내 안을 깊이 들여다보면, 어두운 거울 속에서 운명의 형상들이 잠들어 있는 곳을 바라보면, 그 검은 거울 위로 고개를 숙이는 것만으로도 내 모습을 볼 수 있다. 이제 그 모습은 내 친구이자 길잡이인 그를 완전히 닮아 있다.

**[출처 3 | PDF 130쪽]**

…그것이 무엇인지. 나는 창문 여기저기에 불이 켜진 매춘업소들이 있는 변두리를 지나갔다. 더 바깥쪽에는 새로 짓는 건물들과, 잿빛 눈이 일부 덮인 벽돌 더미가 있었다.

이 기묘한 충동에 이끌려 몽유병자처럼 돌아다니던 나는 고향의 신축 건물을 떠올렸다. 나를 괴롭히던 Kromer가 처음 돈 문제를 정산하려고 나를 데려갔던 곳이었다.

잿빛 밤 속에서 그와 비슷한 건물이 내 앞에 서 있었다. 활짝 벌어진 출입구가 나를 향해 입을 벌리고 있었다. 그 건물이 나를 끌어당겼다. 나는 선택의 여지 없이 모래와 쓰레기 더미에 걸려 넘어지며 앞으로 나아갔다. 그 충동은 저항할 수 없었다. 안으로 들어가야만 했다.

나는 널빤지와 깨진 벽돌을 밟으며 황량한 방 안으로 비틀거리며 들어갔다. 축축한 냉기와 돌 냄새가 기분을 가라앉혔다. 모래 더미 하나가 희미한 회색 얼룩처럼 보였고, 그 밖의 모든 것은 어둠 속에 잠겨 있었다.

그때 겁에 질린 목소리가 외쳤다.

“맙소사, Sinclair! 너는 어디서 온 거야?”

어둠 속에서 한 사람이 일어섰다. 유령처럼 작고 마른 소년이었다. 머리카락이 곤두섰다. 나는 같은 학교에 다니는 Knauer임을 알아보았다.

“너는 어떻게 여기까지 온 거야?” 그가 감정에 휩싸여 거의 제정신이 아닌 듯 물었다. “어떻게 나를 찾아낸 거야?”

나는 그가 무슨 뜻으로 말하는지 이해하지 못했다.

“너를 찾고 있었던 건 아니야.” 나는 멍한 상태로 말했다. 한마디 한마디를 하는 데 힘이 들었고, 얼어붙은 듯 무거운 입술 사이로 말을 힘겹게 밀어내야 했다.

그가 나를 빤히 바라보았다.

“나를 찾던 게 아니라고?”

“응. 이곳에 끌려왔어. 네가 나를 불렀니? 분명 네가 나를 불렀을 거야. 그런데 여기서 뭘 하고 있어? 한밤중이잖아.”

6. 검색 설정 비교

In [42]:
comparison_question = "How is Demian's face described?"
similarity_retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

comparison_results = {}
for name, retriever_to_check in [
    ("기존 MMR, k=3", qa.retriever),
    ("비교 similarity, k=3", similarity_retriever),
]:
    print(f"\n=== {name} ===")
    docs = retriever_to_check.invoke(comparison_question)
    comparison_results[name] = docs
    for i, doc in enumerate(docs, 1):
        print(f"\n[결과 {i} | PDF {doc.metadata['page'] + 1}쪽]")
        print(doc.page_content)


=== 기존 MMR, k=3 ===

[결과 1 | PDF 54쪽]
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
d

MMR과 similarity(유사도)의 검색 결과는 **내용과 순서가 모두 같아 한 번만 번역했습니다.** 텍스트 추출 과정의 오탈자는 문맥에 맞게 읽었고, 중간에 끊긴 부분은 ‘…’로 표시했습니다.

**[결과 1 | PDF 54쪽]**

구역질이 나는 가운데, 나는 Demian의 표정을 살펴보았다. 그는 앞쪽으로 나서지 않고 맨 뒤에 서 있었으며, 평소처럼 품위 있고 느긋해 보였다. 그의 시선은 말의 머리를 향하는 듯했다. 그 눈빛에는 다시금 깊고 고요한 몰입이 드러나 있었다. 거의 광적이면서도 열정적인 몰입이었다.

나는 한동안 그를 바라보지 않을 수 없었다. 바로 그때, 의식의 아득한 곳에서 무척 기이한 느낌이 일어나는 것을 느꼈다. Demian의 얼굴을 보며, 그것이 소년의 얼굴이 아니라 어른 남자의 얼굴이라는 생각이 들었다. 그러다가 그것이 사실은 어른 남자의 얼굴도 아니라는 것을 보았다. 아니, 깨달았다고 해야 할 것이다. 그 얼굴에는 무언가 다른 것이, 거의 여성적인 요소가 깃들어 있었다.

그 순간 그의 얼굴은 남성적이지도, 어린아이 같지도 않았다. 늙지도 젊지도 않았지만 백 년쯤 된 듯했고, 거의 시간을 초월한 채 우리가 사는 시대와는 다른 역사적 시대들의 흔적을 지니고 있었다. 동물이나 나무, 별들이 그런 모습을 하고 있을지도 몰랐다.

물론 당시의 나는 그런 것을 알지 못했다. 어른이 된 지금 이렇게 쓰고 있는 감정을 그때 정확히 느낀 것은 아니었지만, 그와 비슷한 무엇인가를 느꼈다. 어쩌면 그는 잘생겼고, 나는 그에게 매력을 느꼈을지도 모른다. 동시에 그에게 거부감을 느꼈을 수도 있다. 그조차 확신할 수 없었다.

내가 알 수 있었던 것은 그가 우리와 다르다는 사실뿐이었다. 그는 동물이나 영혼, 혹은 어떤 형상과도 같았다. 그가 우리와 달랐다는 것, 상상할 수 없을 만큼 달랐다는 것 외에는 그를 설명할 수 없다. 그 밖의 모든 일에 대한 기억은 텅 비어 있다. 어쩌면 지금 묘사한 것조차 어느 정도는 나중에 받은 인상에서 끌어온 것인지도 모른다.

내가 그와 가까이 지내게 된 것은 몇 년 더 나이가 든 뒤였다. 일반적인 관례와 달리, Demian은 다른 소년들과 함께 견진성사를 받지 않았는데…

**[결과 2 | PDF 90쪽]**

왜 이제야 깨달았을까! 그것은 Demian의 얼굴이었다.

그 뒤로 나는 종이 위의 얼굴을, 기억 속에 남아 있는 Demian의 얼굴과 자주 비교했다. 두 얼굴은 분명 닮았지만 똑같지는 않았다. 그래도 그것이 Demian이라는 데에는 조금도 의심의 여지가 없었다.

초여름의 어느 저녁, 붉은 햇빛이 서쪽을 향한 내 창문으로 비스듬히 들어오고 있었다. 방 안은 어스름했다. 나는 Beatrice의 그림, 혹은 Demian의 그림을 창살에 붙여 놓고 햇빛이 비칠 때 어떻게 보이는지 살펴보고 싶어졌다.

얼굴의 윤곽은 흐릿해졌지만, 분홍빛으로 테두리가 물든 눈과 환한 이마, 생기 넘치는 붉은 입술은 그림 표면에서 가슴을 설레게 할 만큼 빛났다. 그림이 어둠 속으로 사라진 뒤에도 나는 오랫동안 그 맞은편에 앉아 있었다.

그러면서 점차 그 얼굴이 Beatrice도 Demian도 아닌, 나 자신이라는 느낌이 들었다. 그림이 나를 닮았다는 뜻은 아니었다. 나는 그것이 나와 닮아야 한다고 생각하지도 않았다. 하지만 그 얼굴은 어쩐지 내 삶을 표현하고 있었다. 그것은 내 내면의 자아였고, 내 운명이었으며, 내 다이몬이었다.

언젠가 내 친구를 다시 만나게 된다면 그는 저런 모습일 것이다. 내가 사랑하는 여인도, 언젠가 내게 연인이 생긴다면 저런 모습일 것이다. 그것은 내 삶과 죽음의 형상이었다. 내 운명의 음조와 리듬을 표현하고 있었다.

그 몇 주 동안 나는 이전에 읽었던 어떤 책보다도 깊은 인상을 남기는 책을 읽기 시작했다. 훗날에도 그렇게 완전히 책에 빠져든 적은 드물었다. 어쩌면 Nietzsche의 책들조차 예외가 아닐 것이다. 그것은 Novalis의 책 한 권으로…

*역자 주: 소설 『Demian』은 처음에 Emil Sinclair라는 필명으로 출간되었다. 이 이름은 Hesse가 무척 존경했던 시인 Novalis의 친구 이름이다.*

**[결과 3 | PDF 88쪽]**

…

거리에서 가끔 마주치는 그 소녀의 얼굴을 담아내려고 애쓸수록, 그림은 뜻대로 되지 않았다. 마침내 나는 그 시도를 포기했다. 대신 상상 속에서 떠오르는 얼굴을 그리는 것으로 만족했다. 붓을 물감에 담글 때마다 저절로 떠오르는 생각을 따라 그렸다.

그렇게 나타난 것은 꿈속의 얼굴이었다. 나는 그 얼굴이 마음에 들지 않았다. 그래도 실험을 계속했고, 새로 그릴 때마다 내가 찾는 모습에 조금씩 가까워졌다. 여전히 실제 모습과는 거리가 멀었지만 말이다.

나는 점점 더, 특별한 모델을 떠올리지 않은 채 무심히 연필로 선을 긋고 색을 칠하는 일에 익숙해졌다. 오직 무의식에서 종이 위로 흘러나와, 장난 반 진심 반으로 그은 선들 속에서 형태를 갖추는 것들을 따라갔다.

그러던 어느 날, 거의 나도 모르는 사이에 이전 그림들보다 더 뚜렷한 무언가를 표현하는 초상화를 그렸다. 그것은 그 소녀의 얼굴이 아니었다. 이미 오래전부터 소녀의 얼굴에서 벗어나, 전혀 다른 무엇이 되어 있었다. 현실에는 없는 모습이었지만, 내게는 그 소녀만큼이나 큰 의미가 있었다.

그 얼굴은 소녀보다는 소년의 얼굴에 가까웠다. 머리카락은 내가 아름답다고 생각한 소녀의 밝은 금발과 달리, 붉은 기가 도는 밤색이었다. 턱은 강인하고 결연했으며, 입술은 장밋빛이었다. 전체적으로 다소 굳어 있고 가면 같았지만, 강한 인상을 주었고 은밀한 내면의 삶을 암시하고 있었다.

완성된 그림 앞에 앉아 있던 나는 기묘한 느낌을 받았다. 그것은 일종의 신상이나 성스러운 가면과도 같았다. 반은 남성이고 반은 여성이었으며, 나이를 가늠할 수 없었다. 뚜렷한 의지를 지니면서도 꿈꾸는 듯했고, 굳어 있으면서도 신비롭게 살아 있었다.

그 얼굴은 내게 무언가를 전하고 있었다. 그것은 내게 속해 있었고, 내게 무언가를 요구하고 있었다. 누군가를 닮았지만, 누구인지는 알 수 없었다.

In [43]:
stored = db.get(include=["documents"])
print("입력 chunk:", len(demian_chunks))
print("Chroma 항목:", len(stored["ids"]))
print("서로 다른 본문:", len(set(stored["documents"])))

입력 chunk: 182
Chroma 항목: 182
서로 다른 본문: 182


In [44]:
comparison_question = "How is Demian's face described?"
k_comparison_results = {}

for k in [1, 3]:
    retriever_to_check = db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

    k_docs = retriever_to_check.invoke(comparison_question)
    k_comparison_results[k] = k_docs

    print(f"\n=== similarity, k={k} ===")
    for i, doc in enumerate(k_docs, 1):
        print(f"\n[결과 {i} | PDF {doc.metadata['page'] + 1}쪽]")
        print(doc.page_content)


=== similarity, k=1 ===

[결과 1 | PDF 54쪽]
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, 

**similarity(유사도), k=1과 k=3의 결과 1은 같은 내용이므로 한 번만 번역했습니다.** 추출 과정의 오탈자는 문맥에 맞게 읽었으며, 끊긴 부분은 ‘…’로 표시했습니다.

**[공통 결과 1 | PDF 54쪽]**

구역질이 나는 가운데, 나는 Demian의 표정을 살펴보았다. 그는 앞쪽으로 나서지 않고 맨 뒤에 서 있었으며, 평소처럼 품위 있고 느긋해 보였다. 그의 시선은 말의 머리를 향하는 듯했다. 그 눈빛에는 다시금 깊고 고요한 몰입이 드러나 있었다. 거의 광적이면서도 열정적인 몰입이었다.

나는 한동안 그를 바라보지 않을 수 없었다. 바로 그때, 의식의 아득한 곳에서 무척 기이한 느낌이 일어나는 것을 느꼈다. Demian의 얼굴을 보며, 그것이 소년의 얼굴이 아니라 어른 남자의 얼굴이라는 생각이 들었다. 그러다가 그것이 사실은 어른 남자의 얼굴도 아니라는 것을 보았다. 아니, 깨달았다고 해야 할 것이다. 그 얼굴에는 무언가 다른 것이, 거의 여성적인 요소가 깃들어 있었다.

그 순간 그의 얼굴은 남성적이지도, 어린아이 같지도 않았다. 늙지도 젊지도 않았지만 백 년쯤 된 듯했고, 거의 시간을 초월한 채 우리가 사는 시대와는 다른 역사적 시대들의 흔적을 지니고 있었다. 동물이나 나무, 별들이 그런 모습을 하고 있을지도 몰랐다.

물론 당시의 나는 그런 것을 알지 못했다. 어른이 된 지금 이렇게 쓰고 있는 감정을 그때 정확히 느낀 것은 아니었지만, 그와 비슷한 무엇인가를 느꼈다. 어쩌면 그는 잘생겼고, 나는 그에게 매력을 느꼈을지도 모른다. 동시에 그에게 거부감을 느꼈을 수도 있다. 그조차 확신할 수 없었다.

내가 알 수 있었던 것은 그가 우리와 다르다는 사실뿐이었다. 그는 동물이나 영혼, 혹은 어떤 형상과도 같았다. 그가 우리와 달랐다는 것, 상상할 수 없을 만큼 달랐다는 것 외에는 그를 설명할 수 없다. 그 밖의 모든 일에 대한 기억은 텅 비어 있다. 어쩌면 지금 묘사한 것조차 어느 정도는 나중에 받은 인상에서 끌어온 것인지도 모른다.

내가 그와 가까이 지내게 된 것은 몇 년 더 나이가 든 뒤였다. 일반적인 관례와 달리, Demian은 다른 소년들과 함께 견진성사를 받지 않았는데…

---

**[similarity, k=3 — 결과 2 | PDF 90쪽]**

왜 이제야 깨달았을까! 그것은 Demian의 얼굴이었다.

그 뒤로 나는 종이 위의 얼굴을, 기억 속에 남아 있는 Demian의 얼굴과 자주 비교했다. 두 얼굴은 분명 닮았지만 똑같지는 않았다. 그래도 그것이 Demian이라는 데에는 조금도 의심의 여지가 없었다.

초여름의 어느 저녁, 붉은 햇빛이 서쪽을 향한 내 창문으로 비스듬히 들어오고 있었다. 방 안은 어스름했다. 나는 Beatrice의 그림, 혹은 Demian의 그림을 창살에 붙여 놓고 햇빛이 비칠 때 어떻게 보이는지 살펴보고 싶어졌다.

얼굴의 윤곽은 흐릿해졌지만, 분홍빛으로 테두리가 물든 눈과 환한 이마, 생기 넘치는 붉은 입술은 그림 표면에서 가슴을 설레게 할 만큼 빛났다. 그림이 어둠 속으로 사라진 뒤에도 나는 오랫동안 그 맞은편에 앉아 있었다.

그러면서 점차 그 얼굴이 Beatrice도 Demian도 아닌, 나 자신이라는 느낌이 들었다. 그림이 나를 닮았다는 뜻은 아니었다. 나는 그것이 나와 닮아야 한다고 생각하지도 않았다. 하지만 그 얼굴은 어쩐지 내 삶을 표현하고 있었다. 그것은 내 내면의 자아였고, 내 운명이었으며, 내 다이몬이었다.

언젠가 내 친구를 다시 만나게 된다면 그는 저런 모습일 것이다. 내가 사랑하는 여인도, 언젠가 내게 연인이 생긴다면 저런 모습일 것이다. 그것은 내 삶과 죽음의 형상이었다. 내 운명의 음조와 리듬을 표현하고 있었다.

그 몇 주 동안 나는 이전에 읽었던 어떤 책보다도 깊은 인상을 남기는 책을 읽기 시작했다. 훗날에도 그렇게 완전히 책에 빠져든 적은 드물었다. 어쩌면 Nietzsche의 책들조차 예외가 아닐 것이다. 그것은 Novalis의 책 한 권으로…

*역자 주: 소설 『Demian』은 처음에 Emil Sinclair라는 필명으로 출간되었다. 이 이름은 Hesse가 무척 존경했던 시인 Novalis의 친구 이름이다.*

---

**[similarity, k=3 — 결과 3 | PDF 88쪽]**

…

거리에서 가끔 마주치는 그 소녀의 얼굴을 담아내려고 애쓸수록, 그림은 뜻대로 되지 않았다. 마침내 나는 그 시도를 포기했다. 대신 상상 속에서 떠오르는 얼굴을 그리는 것으로 만족했다. 붓을 물감에 담글 때마다 저절로 떠오르는 생각을 따라 그렸다.

그렇게 나타난 것은 꿈속의 얼굴이었다. 나는 그 얼굴이 마음에 들지 않았다. 그래도 실험을 계속했고, 새로 그릴 때마다 내가 찾는 모습에 조금씩 가까워졌다. 여전히 실제 모습과는 거리가 멀었지만 말이다.

나는 점점 더, 특별한 모델을 떠올리지 않은 채 무심히 연필로 선을 긋고 색을 칠하는 일에 익숙해졌다. 오직 무의식에서 종이 위로 흘러나와, 장난 반 진심 반으로 그은 선들 속에서 형태를 갖추는 것들을 따라갔다.

그러던 어느 날, 거의 나도 모르는 사이에 이전 그림들보다 더 뚜렷한 무언가를 표현하는 초상화를 그렸다. 그것은 그 소녀의 얼굴이 아니었다. 이미 오래전부터 소녀의 얼굴에서 벗어나, 전혀 다른 무엇이 되어 있었다. 현실에는 없는 모습이었지만, 내게는 그 소녀만큼이나 큰 의미가 있었다.

그 얼굴은 소녀보다는 소년의 얼굴에 가까웠다. 머리카락은 내가 아름답다고 생각한 소녀의 밝은 금발과 달리, 붉은 기가 도는 밤색이었다. 턱은 강인하고 결연했으며, 입술은 장밋빛이었다. 전체적으로 다소 굳어 있고 가면 같았지만, 강한 인상을 주었고 은밀한 내면의 삶을 암시하고 있었다.

완성된 그림 앞에 앉아 있던 나는 기묘한 느낌을 받았다. 그것은 일종의 신상이나 성스러운 가면과도 같았다. 반은 남성이고 반은 여성이었으며, 나이를 가늠할 수 없었다. 뚜렷한 의지를 지니면서도 꿈꾸는 듯했고, 굳어 있으면서도 신비롭게 살아 있었다.

그 얼굴은 내게 무언가를 전하고 있었다. 그것은 내게 속해 있었고, 내게 무언가를 요구하고 있었다. 누군가를 닮았지만, 누구인지는 알 수 없었다.

## RAG 파이프라인 실행 결과 및 검증

### 질문별 답변과 출처의 일치 여부

#### 질문 1. Demian은 어떻게 생겼나요?

- **반환된 출처:** PDF 54·71·128쪽
- **답변 요약:** 품위 있고 여유로우며, 갈색 머리와 시간을 초월한 듯한 얼굴을 지녔다. 때로는 창백하고 움직임 없는 얼굴이 돌가면처럼 보인다.
- **검증:** 54쪽은 여유로운 태도와 나이·성별을 쉽게 규정하기 어려운 인상을 뒷받침한다. 71쪽에는 창백한 돌가면 같은 얼굴, 움직이지 않는 모습, 갈색 머리가 직접 등장한다. 128쪽은 Demian을 닮은 그림에 관한 내용으로, 실제 외모를 설명하는 데에는 간접적인 근거다.
- **판단:** 답변의 주요 내용이 54·71쪽의 묘사와 일치한다. 검색된 문서마다 근거로서의 직접성에는 차이가 있다.

#### 질문 2. Damian은 왜 사과를 훔쳤다는 이야기를 했나요?

- **반환된 출처:** PDF 13·30·37쪽
- **답변 요약:** 사과를 훔쳤다고 말한 사람을 Sinclair로 정정하고, Kromer가 그 말을 이용해 협박했다고 설명했다.
- **검증:** 13쪽에는 실제로 도둑질하지 않았음에도 훔쳤다고 맹세한 상황과 이를 이용한 협박이 나온다. 30쪽은 Cain 이야기의 해석, 37쪽은 Kromer의 다른 요구를 다루므로, 사과 이야기를 지어낸 동기를 직접 설명하는 근거로는 부족하다.
- **판단:** 거짓 주장과 협박에 관한 설명은 출처와 부합한다. 다만 질문의 ‘왜’에 해당하는 동기는 충분히 설명하지 못했다.

#### 질문 3. Sinclair는 Demian을 처음 어떻게 만나나요?

- **반환된 출처:** PDF 146·182·130쪽
- **답변 요약:** 제공된 발췌문만으로는 첫 만남을 알 수 없다고 답했다.
- **검증:** 146쪽은 Demian과 다시 만나는 장면이며, 182쪽은 작품 후반의 장면이다. 130쪽에서 만나는 인물은 Knauer다. 반환된 문서에는 첫 만남을 설명할 직접적인 근거가 없다.
- **판단:** 필요한 장면을 찾지 못한 검색 실패 사례다. 답변을 유보한 판단은 적절하다. 다만 반환된 문서 전체를 ‘Demian과 재회하는 장면’으로 묶은 설명은 정확성이 부족하다.





### 검색 문서 수(k)에 따른 비교

같은 질문·벡터 저장소·검색 방식(`similarity`)을 유지하고, `k`만 1과 3으로 변경했다.

**질문:** How is Demian's face described?

| 설정 | 검색된 PDF 페이지 | 근거의 특징 |
|---|---|---|
| `k=1` | 54쪽 | Demian의 얼굴과 인상을 직접 묘사한다. |
| `k=3` | 54·90·88쪽 | 직접적인 외모 묘사에 초상화와 상징적 얼굴 묘사가 추가된다. |

두 설정의 첫 번째 문서는 동일했다. 54쪽에는 Demian의 얼굴이 성숙하고, 성별과 나이를 쉽게 규정하기 어려우며, 시간을 초월한 듯하다는 묘사가 있다.

`k=3`에서 추가된 90쪽은 그림 속 얼굴을 Demian과 비교하면서 점차 Sinclair 자신의 내면과 연결하는 내용이다. 88쪽은 상상으로 그린 초상화의 머리색·턱·입술과 상징적인 인상을 묘사한다. 두 문서는 얼굴의 상징성을 이해하는 데 도움이 되지만, 초상화의 특징을 Demian의 실제 외모로 받아들이지 않도록 구분해야 한다.

**설정 선택:** 이 질문에서 직접적인 외모 묘사에 집중한다는 기준으로 `k=1`을 선택한다. 핵심 근거인 54쪽을 유지하면서 참고 문서의 양을 줄일 수 있기 때문이다. 초상화와 인물의 상징적 관계까지 설명하려는 질문에서는 `k=3`의 추가 문맥이 유용할 수 있다.

**비교의 한계:** 이번 실험은 질문 하나에 대한 검색 결과를 비교했다. 설정별로 LLM 답변을 새로 생성하지 않았으므로, 답변 정확도 향상까지 확인한 결과는 아니다.

In [46]:
scored_docs = db.similarity_search_with_relevance_scores(
    comparison_question,
    k=3,
)

# 임계값을 적용하기 전의 검색 점수
for doc, score in scored_docs:
    print(f"PDF {doc.metadata['page'] + 1}쪽 | 점수: {score:.4f}")

# 같은 후보에 임계값을 달리 적용
for threshold in [0.5, 0.4, 0.3]:
    passed_pages = [
        doc.metadata["page"] + 1
        for doc, score in scored_docs
        if score >= threshold
    ]
    print(f"임계값 {threshold}: {len(passed_pages)}개, {passed_pages}")

PDF 54쪽 | 점수: 0.4187
PDF 90쪽 | 점수: 0.3953
PDF 88쪽 | 점수: 0.3876
임계값 0.5: 0개, []
임계값 0.4: 1개, [54]
임계값 0.3: 3개, [54, 90, 88]


### 관련성 점수 임계값 비교

같은 질문의 상위 3개 검색 문서에 서로 다른 임계값을 적용했다.

| PDF 페이지 | 관련성 점수 | 내용 |
|---|---:|---|
| 54쪽 | 0.4187 | Demian의 얼굴과 인상을 직접 묘사 |
| 90쪽 | 0.3953 | 초상화를 Demian과 Sinclair의 내면에 연결 |
| 88쪽 | 0.3876 | 상상으로 그린 초상화의 특징과 상징성 |

| 임계값 | 통과한 페이지 | 해석 |
|---|---|---|
| 0.5 | 없음 | 필요한 근거인 54쪽까지 제외됐다. |
| 0.4 | 54쪽 | 직접적인 외모 묘사를 유지했다. |
| 0.3 | 54·90·88쪽 | 초상화와 상징적 묘사까지 포함했다. |

**설정 선택:** 이번 질문에서는 직접적인 외모 묘사에 집중하기 위해
임계값 0.4를 선택한다. 필요한 근거를 유지하면서 간접적인 문맥을
줄일 수 있었기 때문이다.

**한계:** 90쪽의 점수 0.3953은 기준값 0.4와 가깝다.
따라서 0.4가 모든 질문에 적합한 기준이라고 일반화할 수는 없다.
이번 실험은 검색 문서의 필터링 결과를 비교했으며,
설정별 LLM 답변의 정확도는 평가하지 않았다.